In [6]:
import numpy as np
import matplotlib.pyplot as plt

In [7]:
## For this segment of the code I am submitting it starts off by creating and evaluating a coordinate system to represent the nodes provided in the 
## problem. Then it calculates the resistance of each member. Through the geometry it is able to solve for the length and direction of movement within
## the member. Then with k_global there is the formation of  6*6 matrix which is representative of the 3 nodes seen in the problem. Finally, before 
## solving there is a section of code that is meant to solve for the boundry and loading conditions of the problem. 

In [8]:
def solve_complete_truss(L, EA, F3, F4):
    # 1. Geometry and Connectivity (Figure E2-1)
    # Node 1: (0,0), Node 2: (L,0), Node 3: (0,L)
    nodes = np.array([[0, 0], [L, 0], [0, L]])
    
    # Elements: [Node_Start, Node_End]
    # Element 1: 1-2, Element 2: 2-3, Element 3: 3-1
    elements = [[0, 1], [1, 2], [2, 0]]
    
    num_nodes = len(nodes)
    num_dof = 2 * num_nodes
    K_global = np.zeros((num_dof, num_dof))
    
    # 2. Global Stiffness Matrix Assembly
    for i, (n1, n2) in enumerate(elements):
        x1, y1 = nodes[n1]
        x2, y2 = nodes[n2]
        
        Length = np.sqrt((x2 - x1)**2 + (y2 - y1)**2)
        c = (x2 - x1) / Length
        s = (y2 - y1) / Length
        
        # Element stiffness matrix in global coordinates
        ke = (EA / Length) * np.array([
            [c*c,  c*s, -c*c, -c*s],
            [c*s,  s*s, -c*s, -s*s],
            [-c*c, -c*s,  c*c,  c*s],
            [-c*s, -s*s,  c*s,  s*s]
        ])
        
        # Map to global degrees of freedom
        dofs = [2*n1, 2*n1+1, 2*n2, 2*n2+1]
        for ix, row in enumerate(dofs):
            for jx, col in enumerate(dofs):
                K_global[row, col] += ke[ix, jx]

    # 3. Apply Boundary Conditions and Loads
    # Loads at Node 2: F3 (x-direction), F4 (y-direction)
    F = np.zeros(num_dof)
    F[2] = F3 # u3 index
    F[3] = F4 # u4 index
    
    # Fixed Nodes 1 and 3 (u1=u2=u5=u6=0)
    free_dofs = [2, 3] # Only Node 2 is free to move
    
    # 4. Solve for Displacements at Node 2
    K_red = K_global[np.ix_(free_dofs, free_dofs)]
    F_red = F[free_dofs]
    U_free = np.linalg.solve(K_red, F_red)
    
    # Full displacement vector
    U = np.zeros(num_dof)
    U[free_dofs] = U_free
    
    return U, elements, nodes

# --- PLUGIN VALUES ---
L_val = 1.0       # Length L
EA_val = 210e6    # Example EA
F3_val = 1000.0   # Horizontal force at Node 2
F4_val = 500.0    # Vertical force at Node 2

U_field, elements, nodes = solve_complete_truss(L_val, EA_val, F3_val, F4_val)

print("Displacement Field {u}:")
for i in range(len(U_field)):
    print(f"u{i+1} = {U_field[i]:.6e}")
    

Displacement Field {u}:
u1 = 0.000000e+00
u2 = 0.000000e+00
u3 = 7.142857e-06
u4 = 1.387721e-05
u5 = 0.000000e+00
u6 = 0.000000e+00


In [9]:
## Below is a version of the code that formats the answer into a matrix like that in the textbook. The answers correspond to the textbook as Element 1
## is the sum of the F3 and F4 values found in the previous set of code. Aso, we can see that with element 3 both this generated code and that of the 
## bood is showing all zeros. 
## This section of code relys on the previous on above. In this  section of code
## each element has its own subsection of code. For all three elements the code applies a matrix that represents the amount of movement that particular
## member can withstand. For instance, if it is fixed at one or both of its nodes. Then using the equation EA/L the code solves for the element. For 
## the other two members the code must add an element of rotation represented in a rotation matrix to adjust for the angle represented in the problem. 
## Then is adjusts the presentations and execution of the equation to match the combination of the rotation and local stiffness that was solved for 
## earlier. 

In [10]:
def calculate_book_forces(U_full, EA, L):
    u1, u2, u3, u4, u5, u6 = U_full
    
    # Element 1: Nodes 1-2 (Horizontal)
    # Matching the book's matrix for Element 1
    T1 = np.array([[1, 0, -1, 0], [0, 0, 0, 0], [-1, 0, 1, 0], [0, 0, 0, 0]])
    u_elem1 = np.array([u1, u2, u3, u4])
    P_elem1 = (EA / L) * T1 @ u_elem1
    
    # Element 2: Nodes 2-3 (Diagonal)
    # Matching the book's matrix for Element 2
    c, s = -1/np.sqrt(2), 1/np.sqrt(2)
    T2_rot = np.array([[c, -s, 0, 0], [s, c, 0, 0], [0, 0, c, -s], [0, 0, s, c]])
    k_local = np.array([[1, 0, -1, 0], [0, 0, 0, 0], [-1, 0, 1, 0], [0, 0, 0, 0]])
    u_elem2 = np.array([u3, u4, u5, u6])
    # The book simplifies this to a local-to-global relation
    P_elem2 = (EA / (np.sqrt(2)*L)) * k_local @ T2_rot @ u_elem2
    
    # Element 3: Nodes 3-1 (Vertical)
    # Matching the book's matrix for Element 3
    T3 = np.array([[1, 0, -1, 0], [0, 0, 0, 0], [-1, 0, 1, 0], [0, 0, 0, 0]])
    rot3 = np.array([[0, 1, 0, 0], [-1, 0, 0, 0], [0, 0, 0, 1], [0, 0, -1, 0]])
    u_elem3 = np.array([u1, u2, u5, u6])
    P_elem3 = (EA / L) * T3 @ rot3 @ u_elem3
    
    return P_elem1, P_elem2, P_elem3

p1, p2, p3 = calculate_book_forces(U_field, EA_val, L_val)
print(f"Element 1 Forces: {p1}")
print(f"Element 2 Forces: {p2}")
print(f"Element 3 Forces: {p3}")

Element 1 Forces: [-1500.     0.  1500.     0.]
Element 2 Forces: [-2207.10678119     0.          2207.10678119     0.        ]
Element 3 Forces: [0. 0. 0. 0.]


In [11]:
## The code below is adjusted to allow for a modular use of the 2.1 example. So if we wanted to change the geometry then we would want to change
## the section of code under #1 define coordinates, where we could change the L value to reach the desired geometry updates that we want. Additionally 
## we can change the type of supports, for example from a pin to roller or vice versu by editing the bcs dictionary. Finally we can change the load case by
## adjusting the forces dictionary. 
## The code below is first 
## represented by creating the global stiffness matrix Assembly then setting up the forces and boundary conditions. From here it moves on to solve the 
## displacement field. This involves the spring equation F=Ku. Finally it solves for the Internal forces before inputting the code into the print function
## for the user interface.

In [12]:
def universal_truss_solver(nodes, elements, properties, bcs, forces):
    num_nodes = len(nodes)
    num_dof = 2 * num_nodes
    K_global = np.zeros((num_dof, num_dof))
    
    # 1. Global Stiffness Matrix Assembly
    for i, (n1, n2) in enumerate(elements):
        EA = properties[i]
        x1, y1 = nodes[n1]
        x2, y2 = nodes[n2]
        
        L = np.sqrt((x2 - x1)**2 + (y2 - y1)**2)
        c = (x2 - x1) / L
        s = (y2 - y1) / L
        
        # Element stiffness matrix (Global)
        ke = (EA / L) * np.array([
            [c*c,  c*s, -c*c, -c*s],
            [c*s,  s*s, -c*s, -s*s],
            [-c*c, -c*s,  c*c,  c*s],
            [-c*s, -s*s,  c*s,  s*s]
        ])
        
        dofs = [2*n1, 2*n1+1, 2*n2, 2*n2+1]
        for ix, row in enumerate(dofs):
            for jx, col in enumerate(dofs):
                K_global[row, col] += ke[ix, jx]

    # 2. Setup Forces and Boundary Conditions
    F = np.zeros(num_dof)
    for node_idx, force_vec in forces.items():
        F[2*node_idx] = force_vec[0]
        F[2*node_idx+1] = force_vec[1]
        
    free_dofs = np.ones(num_dof, dtype=bool)
    for node_idx, fixed in bcs.items():
        if fixed[0] is not None: free_dofs[2*node_idx] = False
        if fixed[1] is not None: free_dofs[2*node_idx+1] = False
    
    # 3. Solve for Displacement Field
    U = np.zeros(num_dof)
    K_reduced = K_global[np.ix_(free_dofs, free_dofs)]
    F_reduced = F[free_dofs]
    U[free_dofs] = np.linalg.solve(K_reduced, F_reduced)
    
    # 4. Calculate Internal Forces for each Element
    member_forces = []
    U_reshaped = U.reshape(-1, 2)
    for i, (n1, n2) in enumerate(elements):
        EA = properties[i]
        x1, y1 = nodes[n1]
        x2, y2 = nodes[n2]
        L = np.sqrt((x2 - x1)**2 + (y2 - y1)**2)
        c, s = (x2 - x1) / L, (y2 - y1) / L
        
        u_local = np.array([U_reshaped[n1][0], U_reshaped[n1][1], 
                           U_reshaped[n2][0], U_reshaped[n2][1]])
        T = np.array([-c, -s, c, s])
        force = (EA / L) * np.dot(T, u_local)
        member_forces.append(force)
        
    return U, member_forces

# --- TEST VARIATIONS HERE ---

# 1. Define Coordinates [x, y]
L = 1.0
nodes = np.array([
    [0, 0], # Node 1
    [L, 0], # Node 2
    [0, L]  # Node 3
])

# 2. Define Connectivity [StartNode, EndNode]
elements = [[0, 1], [1, 2], [2, 0]]

# 3. Define EA for each element
EA_val = 1e6
properties = [EA_val, EA_val, EA_val]

# 4. Define Boundary Conditions {Node: [X, Y]} (0=Fixed, None=Free)
bcs = {0: [0, 0], 2: [0, 0]}

# 5. Define Applied Forces {Node: [Fx, Fy]}
forces = {1: [1000, 500]} # F3=1000, F4=500

# RUN
U_result, P_result = universal_truss_solver(nodes, elements, properties, bcs, forces)

print("--- Displacements ---")
for i in range(len(U_result)//2):
    print(f"Node {i+1}: ux={U_result[2*i]:.4e}, uy={U_result[2*i+1]:.4e}")

print("\n--- Internal Axial Forces (P) ---")
for i, f in enumerate(P_result):
    state = "Tension" if f > 1e-9 else ("Compression" if f < -1e-9 else "Zero-Force")
    print(f"Element {i+1}: {f:.2f} ({state})")

--- Displacements ---
Node 1: ux=0.0000e+00, uy=0.0000e+00
Node 2: ux=1.5000e-03, uy=2.9142e-03
Node 3: ux=0.0000e+00, uy=0.0000e+00

--- Internal Axial Forces (P) ---
Element 1: 1500.00 (Tension)
Element 2: -707.11 (Compression)
Element 3: 0.00 (Zero-Force)
